# 02 - Data Preparation

This notebook documents and inspects the final ECG preparation pipeline used in the project. It is intentionally exploratory: the goal is not only to say what was done, but to show enough intermediate evidence to make the segmentation and feature dataset auditable.

Main questions:
- What happens to one ECG record during preprocessing?
- How many segments and features are retained in the final artifact?
- Are there missing records/segments, degenerate features, or obvious sample-level issues?
- What does a lightweight sample of the final feature table look like?


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

from config import FINAL_SEGMENT_FEATURES_DIR, WFDB_RECORDS_DIR
from data_loading import load_record
from feature_extraction import extract_segment_features
from preprocessing import preprocess_signal

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)
plt.rcParams["figure.figsize"] = (8, 4)

METADATA_COLUMNS = {
    "patient_id",
    "record_id",
    "segment_id",
    "segment_ref",
    "start_sample",
    "end_sample",
    "label",
    "utility_label",
}


## 1. Sample Record Pipeline

Before looking at the full feature table, inspect one raw ECG record end-to-end. This gives a sanity check for signal shape, sampling rate, lead count, filtering, z-score normalization and segmentation geometry.


In [ ]:
hea_files = sorted(WFDB_RECORDS_DIR.rglob("*.hea"))
sample_record = hea_files[0].with_suffix("")
signal, fs, leads = load_record(sample_record)
processed = preprocess_signal(
    signal=signal,
    fs=fs,
    leads=leads,
    lowcut=0.5,
    highcut=40.0,
    order=4,
    window_sec=2.0,
    step_sec=1.0,
)

sample_record_summary = pd.DataFrame(
    [
        {
            "record": sample_record.name,
            "fs": fs,
            "n_samples": signal.shape[0],
            "duration_sec": signal.shape[0] / fs,
            "n_leads": signal.shape[1],
            "n_segments": processed["segments"].shape[0],
            "segment_shape": processed["segments"].shape[1:],
            "leads": ", ".join(leads),
        }
    ]
)
sample_record_summary


In [ ]:
lead_idx = leads.index("II") if "II" in leads else 0
n_plot = min(int(fs * 2), signal.shape[0])
time_axis = pd.Series(range(n_plot)) / fs

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(time_axis, signal[:n_plot, lead_idx], color="#2a6f97")
axes[0].set_title(f"Raw signal - lead {leads[lead_idx]}")
axes[0].set_ylabel("mV")
axes[1].plot(time_axis, processed["normalized_signal"][:n_plot, lead_idx], color="#b5651d")
axes[1].set_title("Filtered + per-record z-score normalized signal")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("z-score")
plt.tight_layout()
plt.show()


## 2. Segment Geometry

The final Study II configuration uses 2-second windows and 1-second steps. On a 10-second ECG this gives 9 segments per retained record. The table below makes the overlap explicit for the sample record.


In [ ]:
segment_ranges_df = pd.DataFrame(processed["segment_ranges"], columns=["start_sample", "end_sample"])
segment_ranges_df["start_sec"] = segment_ranges_df["start_sample"] / fs
segment_ranges_df["end_sec"] = segment_ranges_df["end_sample"] / fs
segment_ranges_df["duration_sec"] = segment_ranges_df["end_sec"] - segment_ranges_df["start_sec"]
segment_ranges_df


## 3. Example Segment Features

The final model does not receive raw ECG waveforms. It receives handcrafted features extracted per segment and per lead, plus global summaries across leads.


In [ ]:
segment_features = extract_segment_features(
    processed["segments"][0],
    leads=leads,
    fs=fs,
    include_rr_features=False,
)

segment_features_df = pd.DataFrame(
    {
        "feature": list(segment_features.keys()),
        "value": list(segment_features.values()),
    }
)

feature_family_counts = (
    segment_features_df["feature"]
    .str.extract(r"^(lead_[^_]+|global)")[0]
    .fillna("other")
    .value_counts()
    .rename_axis("feature_family")
    .reset_index(name="n_features")
)

display(feature_family_counts)
segment_features_df.head(25)


## 4. Final Dataset Build Audit

This section checks the retained artifact without loading the full dataset into memory. The manifest is enough to verify record count, windowing parameters, chunk structure, and the single failed record.


In [ ]:
manifest = json.loads((FINAL_SEGMENT_FEATURES_DIR / "manifest.json").read_text(encoding="utf-8"))
chunks_df = pd.DataFrame(manifest["chunks"])
errors_path = FINAL_SEGMENT_FEATURES_DIR / manifest.get("errors_file", "errors.csv")
errors_df = pd.read_csv(errors_path) if errors_path.exists() and errors_path.stat().st_size else pd.DataFrame()

expected_segments_per_record = int((10.0 - manifest["window_sec"]) / manifest["step_sec"]) + 1
nominal_segments = manifest["record_count"] * expected_segments_per_record
retained_segments = manifest["total_segments"]

build_audit = pd.DataFrame(
    [
        {"metric": "records listed", "value": manifest["record_count"]},
        {"metric": "window_sec", "value": manifest["window_sec"]},
        {"metric": "step_sec", "value": manifest["step_sec"]},
        {"metric": "expected segments per 10s record", "value": expected_segments_per_record},
        {"metric": "nominal segments", "value": nominal_segments},
        {"metric": "retained segments", "value": retained_segments},
        {"metric": "missing segments", "value": nominal_segments - retained_segments},
        {"metric": "error count", "value": manifest.get("error_count")},
        {"metric": "chunk count", "value": len(chunks_df)},
        {"metric": "elapsed build time (min)", "value": round(manifest.get("elapsed_seconds", 0) / 60, 1)},
    ]
)

display(build_audit)
display(errors_df)


## 5. Lightweight Sample of the Final Feature Table

The full processed dataset is intentionally chunked because loading all rows at once is heavy. For exploratory checks, load a single chunk and inspect structure, labels, feature families, missingness and basic distributions.


In [ ]:
sample_chunk_path = FINAL_SEGMENT_FEATURES_DIR / chunks_df.loc[0, "chunk_file"]
sample_df = pd.read_csv(sample_chunk_path, compression="gzip", low_memory=False)
feature_columns = [col for col in sample_df.columns if col not in METADATA_COLUMNS]
metadata_columns = [col for col in sample_df.columns if col in METADATA_COLUMNS]

sample_table_summary = pd.DataFrame(
    [
        {
            "sample_chunk": sample_chunk_path.name,
            "rows": len(sample_df),
            "columns": sample_df.shape[1],
            "metadata_columns": len(metadata_columns),
            "feature_columns": len(feature_columns),
            "patients_in_chunk": sample_df["patient_id"].nunique() if "patient_id" in sample_df else None,
            "retained_identifiers_in_chunk": sample_df["patient_id"].nunique() if "patient_id" in sample_df else None,
        }
    ]
)

display(sample_table_summary)
display(sample_df[metadata_columns].head())


In [ ]:
label_counts = sample_df["utility_label"].value_counts().rename_axis("utility_label").reset_index(name="segments")
record_segment_counts = sample_df.groupby("patient_id").size().value_counts().sort_index().rename_axis("segments_per_identifier").reset_index(name="n_identifiers")

fig, axes = plt.subplots(1, figsize=(11, 4))
axes.bar(label_counts["utility_label"], label_counts["segments"], color=["#457b9d", "#e76f51"][: len(label_counts)])
axes.set_title("Utility labels in sample chunk")
axes.set_ylabel("Segments")

plt.tight_layout()
plt.show()

display(label_counts)


In [ ]:
def feature_family(feature_name: str) -> str:
    if feature_name.startswith("lead_"):
        parts = feature_name.split("_")
        return "_".join(parts[:2])
    if feature_name.startswith("global_"):
        return "global"
    return "other"

feature_families = pd.Series(feature_columns).map(feature_family).value_counts().rename_axis("feature_family").reset_index(name="n_features")
missing_fraction = sample_df[feature_columns].isna().mean().sort_values(ascending=False).reset_index()
missing_fraction.columns = ["feature", "missing_fraction"]
zero_variance_features = sample_df[feature_columns].var(numeric_only=True).loc[lambda s: s == 0].index.tolist()

display(feature_families)
display(missing_fraction.head(10))
print(f"Zero-variance features in sample chunk: {len(zero_variance_features)}")


In [ ]:
features_to_plot = [
    col for col in ["lead_II_rms", "lead_V1_rms", "global_mean_rms", "global_max_energy"] if col in sample_df.columns
]
if features_to_plot:
    sample_df[features_to_plot].hist(figsize=(11, 6), bins=30, color="#577590")
    plt.suptitle("Selected feature distributions in sample chunk", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Expected plotting features were not found in the sample chunk.")


## Main Takeaways

- The final Study II artifact uses 2-second windows with a 1-second step, giving 9 expected segments per 10-second record.
- The retained dataset contains 406,359 segments; the 9 missing nominal segments correspond to one failed record listed in `errors.csv`.
- Feature extraction produces a 208-feature segment-level representation, dominated by per-lead morphology/statistical features and global cross-lead summaries.
- The notebook deliberately loads only one final chunk for exploratory diagnostics, avoiding the memory issue caused by loading the complete feature table in VS Code.
